<a href="https://colab.research.google.com/github/Debopriyo-Bhowmick/The-CNN-Models-For-Image-Recognization/blob/main/Cats_And_Dogs_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/archive.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPooling2D,Flatten,BatchNormalization,Dropout,GlobalAveragePooling2D

In [ ]:
import os

os.makedirs('/content/data/train/Cat', exist_ok=True)
os.makedirs('/content/data/train/Dog', exist_ok=True)
os.makedirs('/content/data/test/Cat', exist_ok=True)
os.makedirs('/content/data/test/Dog', exist_ok=True)

In [ ]:
import os
import shutil

cat_dir = '/content/PetImages/Cat'
dog_dir = '/content/PetImages/Dog'

cat_files = sorted(os.listdir(cat_dir))
dog_files = sorted(os.listdir(dog_dir))

In [ ]:
for f in cat_files[:8000]:
    shutil.copy(
        os.path.join(cat_dir, f),
        '/content/data/train/Cat'
    )

for f in cat_files[8000:]:
    shutil.copy(
        os.path.join(cat_dir, f),
        '/content/data/test/Cat'
    )

In [ ]:
for f in dog_files[:8000]:
    shutil.copy(
        os.path.join(dog_dir, f),
        '/content/data/train/Dog'
    )

for f in dog_files[8000:]:
    shutil.copy(
        os.path.join(dog_dir, f),
        '/content/data/test/Dog'
    )

In [ ]:
from PIL import Image
import os

num_skipped = 0

for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join("/content/PetImages", folder_name)

    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)

        try:
            img = Image.open(fpath)
            img.verify()
        except:
            num_skipped += 1
            os.remove(fpath)

print("Deleted", num_skipped, "corrupted images")

In [ ]:
#generators used to get the proper data according to the macine
train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/data/train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (256,256)
    )
test_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/data/test',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (256,256)
    )

In [ ]:
from PIL import Image
import os

bad_files = []

for root, dirs, files in os.walk('/content/data'):
    for file in files:
        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.verify()
        except:
            bad_files.append(path)

print("Bad files found:", len(bad_files))

for f in bad_files[:20]:
    print(f)

In [ ]:
# Normalize
def process(image,label):
    image = tf.cast(image/255. ,tf.float32)
    return image,label

train_ds = train_ds.map(process)
test_ds = test_ds.map(process)

In [ ]:
#Create CNN model

model = Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(GlobalAveragePooling2D())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds,epochs=10,validation_data=test_ds)

In [ ]:
import tensorflow as tf
import os

bad_files = []

for root, dirs, files in os.walk('/content/data'):
    for file in files:
        path = os.path.join(root, file)

        try:
            img = tf.io.read_file(path)
            img = tf.io.decode_image(img)
        except Exception:
            bad_files.append(path)

print("Bad files:", len(bad_files))

for f in bad_files[:20]:
    print(f)

In [ ]:
for f in bad_files:
    os.remove(f)

print("Removed", len(bad_files), "bad files")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'],color='red',label='train')
plt.plot(history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],color='red',label='train')
plt.plot(history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
print(train_ds.class_names)

In [ ]:
model.evaluate(test_ds)

In [ ]:
correct = 0

for file in os.listdir('/content/data/test/Dog')[:100]:

    img = cv2.imread('/content/data/test/Dog/' + file)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img,(256,256))

    pred = model.predict(img.reshape(1,256,256,3), verbose=0)[0][0]

    if pred > 0.5:
        correct += 1

print("Dog:", correct)

In [ ]:
correct = 0

for file in os.listdir('/content/data/test/Cat')[:100]:

    img = cv2.imread('/content/data/test/Cat/' + file)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img,(256,256))

    pred = model.predict(img.reshape(1,256,256,3), verbose=0)[0][0]

    if pred < 0.5:
        correct += 1

print("Cat:", correct)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images, labels in test_ds:

    preds = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())

    y_pred.extend(
        (preds > 0.5).astype(int).flatten()
    )

cm = confusion_matrix(y_true, y_pred)

print(cm)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def predict_animal(img_path):

    img = cv2.imread(img_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.axis('off')
    plt.show()

    img = cv2.resize(img, (256,256))

    test_input = img.reshape((1,256,256,3))

    pred = model.predict(test_input, verbose=0)[0][0]

    cat_percent = (1 - pred) * 100
    dog_percent = pred * 100

    print(f"Cat Probability : {cat_percent:.2f}%")
    print(f"Dog Probability : {dog_percent:.2f}%")

    if pred > 0.5:
        print("🐶 Prediction : DOG")
    else:
        print("🐱 Prediction : CAT")

In [ ]:
predict_animal('/content/Cat_s_Mind.webp')

In [ ]:
predict_animal('/content/c529a263-8ffa-4521-a47e-babb146f1e6b.webp')

In [ ]:
predict_animal('/content/images.webp')

In [ ]:
model.save("cat_dog_model.keras")

In [ ]:
import os

print(os.listdir('/content'))

In [ ]:
from google.colab import files

files.download("cat_dog_model.keras")

In [ ]:
!zip -r PetImages.zip /content/PetImages

In [ ]:
from google.colab import files

files.download("PetImages.zip")